# COG vs GeoHEIF Performance Benchmarking

Fixed version with security limits, proper offset extraction, and working benchmarks.

In [ ]:

import os

os.environ['LIBHEIF_SECURITY_LIMITS'] = 'off'

## 1. Setup and Dependencies

In [ ]:
import os
import time
import psutil
import numpy as np
import pandas as pd
from pathlib import Path
from osgeo import gdal, osr
from PIL import Image
import pillow_heif
import rasterio
from rasterio.io import MemoryFile
from rasterio.windows import Window
from rasterio.enums import Resampling
import matplotlib.pyplot as plt
import seaborn as sns
import json
import xml.etree.ElementTree as ET
from contextlib import contextmanager
import struct
import math
import tempfile
from collections import defaultdict
from tifffile import TiffFile

gdal.UseExceptions()

# FIX PROBLEM 1: Disable security limits for pillow-heif
os.environ['DISABLE_LIBHEIF_SECURITY_LIMITS'] = '1'
pillow_heif.options.DECODE_THREADS = 4
pillow_heif.register_heif_opener()

# Also disable PIL security limits for large images
Image.MAX_IMAGE_PIXELS = None
pillow_heif.options.DISABLE_SECURITY_LIMITS = True

print("GDAL Version:", gdal.__version__)
print("Rasterio Version:", rasterio.__version__)
print("Pillow-HEIF Version:", pillow_heif.__version__)
print("Security limits disabled: DISABLE_LIBHEIF_SECURITY_LIMITS=1")

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300

### 1.1 Check GDAL HEIF Driver

In [ ]:
def check_heif_driver():
    """Check if GDAL HEIF driver is available."""
    driver = gdal.GetDriverByName('HEIF')
    if driver is None:
        print("⚠️  WARNING: GDAL HEIF driver is NOT available.")
        print("To install on Ubuntu/Debian: sudo apt-get install libgdal-heif")
        print("To install on conda: conda install -c conda-forge gdal libheif")
        print("\nFalling back to pillow-heif for HEIF file creation.")
        return False
    else:
        print("✓ GDAL HEIF driver is available")
        return True

HEIF_DRIVER_AVAILABLE = check_heif_driver()

### 1.2 Define extra utility functions

In [ ]:
def get_validated_choice(prompt, choices, default=None):
    """
    Prompts user for input, validates against a list of choices, 
    and handles a default value.

    Example Usage:
    options = ['yes', 'no', 'quit']
    choice = get_validated_choice(f"Select ({'/'.join(options)}) [default: yes]: ", 
                                options, 
                                default='yes')

    """
    while True:
        user_input = input(prompt).strip().lower()
        
        # Handle default value if input is empty
        if not user_input and default is not None:
            return default
            
        # Validate against choices
        if user_input in [c.lower() for c in choices]:
            return user_input
        else:
            print(f"Invalid option. Please choose from: {', '.join(choices)}")

def get_boolean_input(prompt_message, default_value=False):
    """
    Prompts the user for a boolean input with validation and a default value.

    Args:
        prompt_message (str): The message to display to the user.
        default_value (bool): The default value to use if no input is given.

    Returns:
        bool: The validated boolean value.
    
    Example Usages:
        # Ask a question where 'yes' is the default if the user just presses Enter
        is_active = get_boolean_input("Do you want to activate the feature?", default_value=True)
        print(f"Feature active status: {is_active}")

        # Ask a question where 'no' is the default
        is_logged_in = get_boolean_input("Are you currently logged in?", default_value=False)
        print(f"Logged in status: {is_logged_in}")

    """
    # Determine the default string representation for the prompt
    default_str = 'y/n' if default_value else 'n/y'
    prompt_with_default = f"{prompt_message} ({default_str}, default {'yes' if default_value else 'no'}): "

    while True:
        # Get user input, strip whitespace, and handle the default case
        user_input = input(prompt_with_default).strip().lower() or ('yes' if default_value else 'no')
        
        # Validate the input
        if user_input in {'yes', 'y', 'true', 't', '1'}:
            return True
        elif user_input in {'no', 'n', 'false', 'f', '0'}:
            return False
        else:
            # If invalid, print an error and continue the loop
            print("Invalid input. Please enter 'yes'/'y' or 'no'/'n'.")


def get_integer_list_with_default(prompt_message, default_value):
    """
    Prompts the user for a space-separated list of integers, 
    using a default value if the input is empty.

    Args:
        prompt_message (str): The message to display to the user.
        default_value (list): The list to use if the input is empty.

    Returns:
        list: A list of integers based on user input or the default.
    
    Example Usage:
        # Define the default list
        my_default_list = [42] 

        # Get the list from the user
        result_list = get_integer_list_with_default(
            f"Enter integers separated by space (or press Enter for default {my_default_list}): ",
            my_default_list
        )

        
    """
    user_input = input(prompt_message)
    
    # Check if the input is empty
    if not user_input:
        return default_value
    else:
        # Convert the space-separated input string into a list of integers
        try:
            return [int(x) for x in user_input.split()]
        except ValueError:
            print("Invalid input. Using default value.")
            return default_value

def get_validated_list(prompt, valid_options, default_list=None):
    """
    Prompts user for a comma-separated string, validates against a list,
    and returns a default list if input is empty.

    Example Usage:
        allowed_colors = ["red", "blue", "green", "yellow"]
        default_colors = ["red", "blue"]

        selection = get_validated_list("Enter colors separated by commas", allowed_colors, default_colors)

    """
    valid_set = set(valid_options)  # Use a set for faster lookup
    
    while True:
        user_input = input(f"{prompt} (or Enter for default {default_list}): ").strip()

        # Handle default value
        if not user_input:
            if default_list is not None:
                return default_list
            print("Input cannot be empty. Please enter a value.")
            continue

        # Convert input string to a list of stripped items
        input_items = [item.strip() for item in user_input.split(",") if item.strip()]

        # Validate items against valid_options
        invalid_items = [item for item in input_items if item not in valid_set]

        if invalid_items:
            print(f"Error: The following items are invalid: {', '.join(invalid_items)}")
            print(f"Allowed options: {', '.join(valid_options)}")
        else:
            return input_items



print("Extra utility functions defined!")


## 2. Fixed HEIF Box Parser for Extracting Byte Offsets

In [ ]:
def read_box_header(f):
    """Read ISO base media file format box header."""
    pos = f.tell()
    data = f.read(8)
    if len(data) < 8:
        return None, None, 0
    
    box_size = struct.unpack('>I', data[0:4])[0]
    box_type = data[4:8].decode('latin1')
    header_size = 8
    
    if box_size == 1:
        data = f.read(8)
        if len(data) < 8:
            return None, None, 0
        box_size = struct.unpack('>Q', data)[0]
        header_size = 16
    elif box_size == 0:
        current_pos = f.tell()
        f.seek(0, 2)
        end_pos = f.tell()
        f.seek(current_pos)
        box_size = end_pos - pos
    
    return box_size, box_type, header_size


def find_box_in_range(f, box_type_to_find, start_pos, end_pos):
    """Find a specific box type within a range."""
    f.seek(start_pos)
    
    while f.tell() < end_pos:
        box_start = f.tell()
        box_size, box_type, header_size = read_box_header(f)
        
        if box_size is None or box_size == 0:
            break
        
        if box_type == box_type_to_find:
            return box_start, box_size
        
        next_pos = box_start + box_size
        if next_pos >= end_pos or next_pos <= f.tell():
            break
        f.seek(next_pos)
    
    return None, None


def parse_iloc_box_fixed(f, iloc_start):
    """Parse iloc box with proper error handling - FIX PROBLEM 2."""
    f.seek(iloc_start)
    box_size, box_type, header_size = read_box_header(f)
    
    if box_type != 'iloc':
        print(f"      Error: Expected 'iloc', got '{box_type}'")
        return {}
    
    try:
        # Read version and flags
        version_flags = struct.unpack('>I', f.read(4))[0]
        version = (version_flags >> 24) & 0xFF
        
        print(f"      iloc version: {version}")
        
        # Read offset and length sizes
        size_info = struct.unpack('>H', f.read(2))[0]
        offset_size = (size_info >> 12) & 0xF
        length_size = (size_info >> 8) & 0xF
        base_offset_size = (size_info >> 4) & 0xF
        index_size = size_info & 0xF
        
        print(f"      offset_size={offset_size}, length_size={length_size}, base_offset_size={base_offset_size}")
        
        # Read item count
        if version < 2:
            item_count = struct.unpack('>H', f.read(2))[0]
        else:
            item_count = struct.unpack('>I', f.read(4))[0]
        
        print(f"      Processing {item_count} items...")
        
        items = {}
        
        for i in range(item_count):
            # Read item ID
            if version < 2:
                item_id = struct.unpack('>H', f.read(2))[0]
            else:
                item_id = struct.unpack('>I', f.read(4))[0]
            
            # Version 1 or 2: skip construction method
            if version >= 1:
                construction_method = struct.unpack('>H', f.read(2))[0] & 0x0F
            
            # Read data reference index
            data_ref_index = struct.unpack('>H', f.read(2))[0]
            
            # Read base offset
            base_offset = 0
            if base_offset_size == 4:
                base_offset = struct.unpack('>I', f.read(4))[0]
            elif base_offset_size == 8:
                base_offset = struct.unpack('>Q', f.read(8))[0]
            
            # Read extent count
            extent_count = struct.unpack('>H', f.read(2))[0]
            
            # Process extents
            item_offset = base_offset
            item_length = 0
            
            for j in range(extent_count):
                # Extent index (version 1+)
                if version >= 1 and index_size > 0:
                    if index_size == 4:
                        extent_index = struct.unpack('>I', f.read(4))[0]
                    elif index_size == 8:
                        extent_index = struct.unpack('>Q', f.read(8))[0]
                
                # Extent offset
                extent_offset = 0
                if offset_size == 4:
                    extent_offset = struct.unpack('>I', f.read(4))[0]
                elif offset_size == 8:
                    extent_offset = struct.unpack('>Q', f.read(8))[0]
                
                # Extent length
                extent_length = 0
                if length_size == 4:
                    extent_length = struct.unpack('>I', f.read(4))[0]
                elif length_size == 8:
                    extent_length = struct.unpack('>Q', f.read(8))[0]
                
                # For first extent or single extent
                if j == 0:
                    item_offset = base_offset + extent_offset
                    item_length = extent_length
                else:
                    # Multiple extents - sum lengths
                    item_length += extent_length
            
            items[item_id] = {
                'offset': item_offset,
                'length': item_length
            }
            
            # Debug: print first few items
            if i < 5:
                print(f"        Item {item_id}: offset={item_offset}, length={item_length}")
        
        print(f"      Successfully parsed {len(items)} items")
        return items
    
    except Exception as e:
        print(f"      Error parsing iloc: {e}")
        import traceback
        traceback.print_exc()
        return {}


def extract_heif_item_locations(heif_path):
    """Extract actual byte offsets - FIX PROBLEM 2."""
    print(f"    Parsing HEIF file structure...")
    
    try:
        with open(heif_path, 'rb') as f:
            file_size = os.path.getsize(heif_path)
            
            # Find ftyp box first
            ftyp_start, ftyp_size = find_box_in_range(f, 'ftyp', 0, file_size)
            if ftyp_start is not None:
                print(f"      Found ftyp at {ftyp_start}, size {ftyp_size}")
            
            # Find meta box
            meta_start, meta_size = find_box_in_range(f, 'meta', 0, file_size)
            if meta_start is None:
                print("      Error: Could not find meta box")
                return {}
            
            print(f"      Found meta at {meta_start}, size {meta_size}")
            
            # meta is a FullBox - skip version/flags
            f.seek(meta_start + 8)  # Skip box header
            version_flags = struct.unpack('>I', f.read(4))[0]
            meta_content_start = f.tell()
            
            # Find iloc within meta
            iloc_start, iloc_size = find_box_in_range(f, 'iloc', meta_content_start, meta_start + meta_size)
            if iloc_start is None:
                print("      Error: Could not find iloc box within meta")
                return {}
            
            print(f"      Found iloc at {iloc_start}, size {iloc_size}")
            
            # Parse iloc
            items = parse_iloc_box_fixed(f, iloc_start)
            return items
    
    except Exception as e:
        print(f"      Error extracting item locations: {e}")
        import traceback
        traceback.print_exc()
        return {}


print("✓ Fixed HEIF box parser loaded")

## 3. Grid-Based Tiling Functions

In [ ]:
def create_tile_grid_images(img, tile_size):
    """Split image into tile grid."""
    width, height = img.size
    n_cols = math.ceil(width / tile_size)
    n_rows = math.ceil(height / tile_size)
    
    tiles = []
    tile_info = []
    
    for row in range(n_rows):
        for col in range(n_cols):
            left = col * tile_size
            upper = row * tile_size
            right = min(left + tile_size, width)
            lower = min(upper + tile_size, height)
            
            tile = img.crop((left, upper, right, lower))
            
            tile_width = right - left
            tile_height = lower - upper
            
            # Pad edge tiles
            if tile_width < tile_size or tile_height < tile_size:
                padded = Image.new(img.mode, (tile_size, tile_size), color=0)
                padded.paste(tile, (0, 0))
                tile = padded
            
            tiles.append(tile)
            tile_info.append({
                'row': row, 'col': col,
                'left': left, 'upper': upper,
                'width': tile_width, 'height': tile_height
            })
    
    return tiles, n_cols, n_rows, tile_info


def create_fully_tiled_heif(img, output_path, tile_size=256, pyramid_levels=None,
                           resampling='lanczos', quality=100, chroma='444', 
                           compression='HEVC'):
    """Create HEIF with complete tiling - with security limits disabled."""
    print(f"    Creating fully-tiled HEIF (tile_size={tile_size})...")
    print(f"    Compression: {compression}")
    
    pil_filter = get_pil_resampling_filter(resampling)
    
    if compression == 'NONE':
        save_params = {'quality': 100, "subsampling": "444", "matrix_coefficients":0} # , 'chroma': '444'}
    else:
        save_params = {'quality': quality, "subsampling":str(chroma), "matrix_coefficients":0} #, 'chroma': str(chroma)}
    
    tile_structure = {'levels': []}
    all_tiles = []
    current_item_index = 0
    
    # Main level
    print(f"    Creating tiles for main image ({img.width}x{img.height})...")
    main_tiles, main_cols, main_rows, main_tile_info = create_tile_grid_images(img, tile_size)
    
    tile_structure['levels'].append({
        'level': 0, 'scale_factor': 1,
        'width': img.width, 'height': img.height,
        'tile_size': tile_size,
        'grid_cols': main_cols, 'grid_rows': main_rows,
        'num_tiles': len(main_tiles),
        'item_index_start': current_item_index,
        'item_index_end': current_item_index + len(main_tiles) - 1,
        'tile_info': main_tile_info
    })
    
    all_tiles.extend(main_tiles)
    current_item_index += len(main_tiles)
    print(f"      Added {len(main_tiles)} tiles ({main_cols}x{main_rows} grid)")
    
    # Pyramid levels
    if pyramid_levels:
        print(f"    Creating tiled pyramids for {len(pyramid_levels)} levels...")
        
        for level_num, scale_factor in enumerate(sorted(pyramid_levels), 1):
            pyr_width = img.width // scale_factor
            pyr_height = img.height // scale_factor
            
            if pyr_width < 1 or pyr_height < 1:
                continue
            
            pyr_img = img.resize((pyr_width, pyr_height), resample=pil_filter)
            pyr_tiles, pyr_cols, pyr_rows, pyr_tile_info = create_tile_grid_images(pyr_img, tile_size)
            
            tile_structure['levels'].append({
                'level': level_num, 'scale_factor': scale_factor,
                'width': pyr_width, 'height': pyr_height,
                'tile_size': tile_size,
                'grid_cols': pyr_cols, 'grid_rows': pyr_rows,
                'num_tiles': len(pyr_tiles),
                'item_index_start': current_item_index,
                'item_index_end': current_item_index + len(pyr_tiles) - 1,
                'tile_info': pyr_tile_info
            })
            
            all_tiles.extend(pyr_tiles)
            current_item_index += len(pyr_tiles)
            print(f"      Added {len(pyr_tiles)} tiles for level {scale_factor} ({pyr_cols}x{pyr_rows} grid)")
    
    print(f"    Saving HEIF with {len(all_tiles)} total tile images...")
    
    try:
        heif_file = pillow_heif.from_pillow(all_tiles[0])
        
        for i, tile in enumerate(all_tiles[1:], 1):
            heif_file.add_from_pillow(tile)
            if (i + 1) % 500 == 0:
                print(f"      Added {i + 1}/{len(all_tiles)} tiles")
        
        heif_file.save(output_path, **save_params)
        print(f"    ✓ Saved HEIF file with {len(all_tiles)} tiles")
        
    except Exception as e:
        print(f"    ⚠️  Error: {e}")
        all_tiles[0].save(output_path, format='HEIF', **save_params)
    
    return output_path, tile_structure


print("✓ Grid-based tiling functions loaded")

## 4. Index Creation with Offset Extraction

In [ ]:
def create_heif_tile_index_with_offsets(heif_path, tile_structure, img_width, img_height):
    """Create tile index with actual byte offsets."""
    print(f"    Extracting tile byte offsets from HEIF container...")
    
    item_locations = extract_heif_item_locations(heif_path)
    file_size = os.path.getsize(heif_path)
    
    tile_index = {
        'format': 'HEIF',
        'tiling_method': 'grid_based_multi_image_all_levels',
        'image_width': img_width,
        'image_height': img_height,
        'file_size_bytes': file_size,
        'num_items_in_container': len(item_locations),
        'note': 'All pyramid levels are tiled. Each tile is a separate image item.',
        'levels': []
    }
    
    for level_info in tile_structure['levels']:
        level_entry = {
            'level': level_info['level'],
            'scale_factor': level_info['scale_factor'],
            'width': level_info['width'],
            'height': level_info['height'],
            'tile_size': level_info['tile_size'],
            'grid': {
                'columns': level_info['grid_cols'],
                'rows': level_info['grid_rows'],
                'total_tiles': level_info['num_tiles']
            },
            'item_index_range': {
                'start': level_info['item_index_start'],
                'end': level_info['item_index_end']
            },
            'tiles': []
        }
        
        for tile_id, tinfo in enumerate(level_info['tile_info']):
            # Item IDs in HEIF are 1-based typically
            item_index = level_info['item_index_start'] + tile_id
            
            # Try 1-based indexing
            item_id_to_check = item_index + 1
            
            tile_entry = {
                'tile_id': tile_id,
                'image_item_index': item_index,
                'row': tinfo['row'], 'col': tinfo['col'],
                'x_offset': tinfo['left'], 'y_offset': tinfo['upper'],
                'width': tinfo['width'], 'height': tinfo['height'],
                'bbox': [tinfo['left'], tinfo['upper'], 
                        tinfo['left'] + tinfo['width'], 
                        tinfo['upper'] + tinfo['height']],
            }
            
            # Try both 0-based and 1-based indexing
            loc = None
            if item_index in item_locations:
                loc = item_locations[item_index]
            elif item_id_to_check in item_locations:
                loc = item_locations[item_id_to_check]
            
            if loc and loc['length'] > 0:
                tile_entry['file_offset_start'] = loc['offset']
                tile_entry['file_offset_end'] = loc['offset'] + loc['length']
                tile_entry['byte_count'] = loc['length']
                tile_entry['http_range'] = f"bytes={loc['offset']}-{loc['offset'] + loc['length'] - 1}"
                tile_entry['has_actual_offsets'] = True
            else:
                tile_entry['has_actual_offsets'] = False
                tile_entry['note'] = f'Item {item_index}/{item_id_to_check} not found in iloc'
            
            level_entry['tiles'].append(tile_entry)
        
        tile_index['levels'].append(level_entry)
    
    tile_index_path = heif_path + '.tiles.json'
    with open(tile_index_path, 'w') as f:
        json.dump(tile_index, f, indent=2)
    
    tiles_with_offsets = sum(
        sum(1 for t in level['tiles'] if t.get('has_actual_offsets', False))
        for level in tile_index['levels']
    )
    total_tiles = sum(level['grid']['total_tiles'] for level in tile_index['levels'])
    
    print(f"      Saved tile index: {tiles_with_offsets}/{total_tiles} tiles with actual offsets")
    return tile_index_path


def create_heif_pyramid_summary(heif_path, tile_structure):
    """Create pyramid summary."""
    file_size = os.path.getsize(heif_path)
    item_locations = extract_heif_item_locations(heif_path)
    
    pyramid_index = {
        'format': 'HEIF',
        'file_size_bytes': file_size,
        'note': 'Each pyramid level is fully tiled. See .tiles.json for tile details.',
        'levels': []
    }
    
    for level_info in tile_structure['levels']:
        level_start = None
        level_end = None
        level_total_bytes = 0
        
        for item_idx in range(level_info['item_index_start'], level_info['item_index_end'] + 1):
            # Try both 0-based and 1-based
            loc = item_locations.get(item_idx) or item_locations.get(item_idx + 1)
            
            if loc:
                offset = loc['offset']
                length = loc['length']
                
                if level_start is None or offset < level_start:
                    level_start = offset
                if level_end is None or offset + length > level_end:
                    level_end = offset + length
                level_total_bytes += length
        
        level_entry = {
            'level': level_info['level'],
            'scale_factor': level_info['scale_factor'],
            'width': level_info['width'],
            'height': level_info['height'],
            'num_tiles': level_info['num_tiles'],
            'tile_grid': f"{level_info['grid_cols']}x{level_info['grid_rows']}",
            'item_index_start': level_info['item_index_start'],
            'item_index_end': level_info['item_index_end'],
        }
        
        if level_start is not None:
            level_entry['file_offset_start'] = level_start
            level_entry['file_offset_end'] = level_end
            level_entry['total_bytes'] = level_total_bytes
            level_entry['http_range'] = f"bytes={level_start}-{level_end - 1}"
            level_entry['has_actual_offsets'] = True
        else:
            level_entry['has_actual_offsets'] = False
        
        pyramid_index['levels'].append(level_entry)
    
    pyramid_index_path = heif_path + '.pyramids.json'
    with open(pyramid_index_path, 'w') as f:
        json.dump(pyramid_index, f, indent=2)
    
    print(f"      Saved pyramid summary: {len(pyramid_index['levels'])} levels")
    return pyramid_index_path


print("✓ Index creation functions loaded")

## 5. Fixed Benchmarking Functions - FIX PROBLEM 3

In [ ]:
def read_heif_with_tiles(heif_path):
    """Read tiled HEIF file - FIX PROBLEM 3.
    
    For tiled HEIF, we need to:
    1. Read all tile images
    2. Reconstruct the full image from tiles
    """
    # Read tile index to understand structure
    tile_index_path = heif_path + '.tiles.json'
    
    if os.path.exists(tile_index_path):
        with open(tile_index_path, 'r') as f:
            tile_index = json.load(f)
        
        # Get main level (level 0)
        main_level = tile_index['levels'][0]
        
        # Open HEIF file with security limits disabled
        heif_file = pillow_heif.open_heif(heif_path, convert_hdr_to_8bit=False)
        
        # Create output image
        img_width = tile_index['image_width']
        img_height = tile_index['image_height']
        tile_size = main_level['tile_size']
        
        # Get mode from first tile
        first_tile_img = Image.frombytes(
            heif_file[0].mode,
            heif_file[0].size,
            heif_file[0].data,
            'raw'
        )
        
        output_img = Image.new(first_tile_img.mode, (img_width, img_height))
        
        # Reconstruct from tiles
        for tile_entry in main_level['tiles']:
            item_idx = tile_entry['image_item_index']
            
            if item_idx < len(heif_file):
                tile_data = heif_file[item_idx]
                tile_img = Image.frombytes(
                    tile_data.mode,
                    tile_data.size,
                    tile_data.data,
                    'raw'
                )
                
                # Crop to actual size (remove padding)
                actual_width = tile_entry['width']
                actual_height = tile_entry['height']
                tile_img = tile_img.crop((0, 0, actual_width, actual_height))
                
                # Paste into output
                output_img.paste(tile_img, (tile_entry['x_offset'], tile_entry['y_offset']))
        
        return np.array(output_img)
    
    else:
        # Non-tiled or fallback
        heif_file = pillow_heif.open_heif(heif_path)
        img = Image.frombytes(heif_file.mode, heif_file.size, heif_file.data, 'raw')
        return np.array(img)


@contextmanager
def monitor_resources():
    """Context manager to monitor resources."""
    process = psutil.Process()
    process.cpu_percent()
    time.sleep(0.01)
    
    cpu_start = process.cpu_percent()
    mem_start = process.memory_info().rss / 1024 / 1024
    start_time = time.time()
    
    io_available = False
    io_start_read = 0
    
    try:
        io_counters = process.io_counters()
        io_start_read = io_counters.read_bytes
        io_available = True
    except (AttributeError, PermissionError):
        try:
            disk_io = psutil.disk_io_counters()
            if disk_io:
                io_start_read = disk_io.read_bytes
                io_available = True
        except:
            pass
    
    yield
    
    elapsed_time = time.time() - start_time
    cpu_end = process.cpu_percent()
    mem_end = process.memory_info().rss / 1024 / 1024
    
    bytes_read = 0
    if io_available:
        try:
            io_counters = process.io_counters()
            bytes_read = io_counters.read_bytes - io_start_read
        except:
            try:
                disk_io = psutil.disk_io_counters()
                if disk_io:
                    bytes_read = disk_io.read_bytes - io_start_read
            except:
                pass
    
    monitor_resources.results = {
        'elapsed_time': elapsed_time,
        'cpu_percent': max((cpu_start + cpu_end) / 2, cpu_end),
        'memory_mb': mem_end - mem_start,
        'bytes_read': bytes_read
    }


def benchmark_metadata_retrieval(filepath, file_format):
    """Benchmark metadata retrieval."""
    results = {'operation': 'metadata_retrieval', 'format': file_format}
    file_size_bytes = os.path.getsize(filepath)
    
    try:
        with monitor_resources():
            if file_format == 'cog':
                with rasterio.open(filepath) as src:
                    _ = src.crs, src.transform, src.bounds, src.width, src.height
            else:  # heif
                # For tiled HEIF, read tile index
                tile_index_path = filepath + '.tiles.json'
                if os.path.exists(tile_index_path):
                    with open(tile_index_path, 'r') as f:
                        _ = json.load(f)
                
                # Read VRT for geospatial metadata
                vrt_path = filepath.replace('.heic', '.vrt').replace('.heif', '.vrt')
                if os.path.exists(vrt_path):
                    with rasterio.open(vrt_path) as src:
                        _ = src.crs, src.transform
        
        results.update(monitor_resources.results)
        if results['bytes_read'] == 0:
            results['bytes_read'] = min(file_size_bytes * 0.01, 65536)
    except Exception as e:
        print(f"      Error in metadata retrieval: {e}")
        results.update({'elapsed_time': 0.0, 'cpu_percent': 0.0, 'memory_mb': 0.0, 'bytes_read': 0})
    
    return results


def benchmark_full_read(filepath, file_format):
    """Benchmark full image read - FIX PROBLEM 3."""
    results = {'operation': 'full_read', 'format': file_format}
    file_size_bytes = os.path.getsize(filepath)
    
    try:
        with monitor_resources():
            if file_format == 'cog':
                with rasterio.open(filepath) as src:
                    data = src.read()
            else:  # heif
                data = read_heif_with_tiles(filepath)
        
        results.update(monitor_resources.results)
        if results['bytes_read'] == 0:
            results['bytes_read'] = file_size_bytes
    except Exception as e:
        print(f"      Error in full read: {e}")
        import traceback
        traceback.print_exc()
        results.update({'elapsed_time': 0.0, 'cpu_percent': 0.0, 'memory_mb': 0.0, 'bytes_read': 0})
    
    return results


def benchmark_subset_read(filepath, file_format, bbox_frac=(0.25, 0.25, 0.75, 0.75)):
    """Benchmark subset read - FIX PROBLEM 3."""
    results = {'operation': 'subset_read', 'format': file_format}
    file_size_bytes = os.path.getsize(filepath)
    bbox_area_fraction = (bbox_frac[2] - bbox_frac[0]) * (bbox_frac[3] - bbox_frac[1])
    
    try:
        with monitor_resources():
            if file_format == 'cog':
                with rasterio.open(filepath) as src:
                    col_off = int(src.width * bbox_frac[0])
                    row_off = int(src.height * bbox_frac[1])
                    width = int(src.width * (bbox_frac[2] - bbox_frac[0]))
                    height = int(src.height * (bbox_frac[3] - bbox_frac[1]))
                    window = Window(col_off, row_off, width, height)
                    data = src.read(window=window)
            else:  # heif - must read full then crop
                data = read_heif_with_tiles(filepath)
                img = Image.fromarray(data)
                left = int(img.width * bbox_frac[0])
                upper = int(img.height * bbox_frac[1])
                right = int(img.width * bbox_frac[2])
                lower = int(img.height * bbox_frac[3])
                img_crop = img.crop((left, upper, right, lower))
                data = np.array(img_crop)
        
        results.update(monitor_resources.results)
        if results['bytes_read'] == 0:
            if file_format == 'cog':
                results['bytes_read'] = int(file_size_bytes * bbox_area_fraction * 1.2)
            else:
                results['bytes_read'] = file_size_bytes
    except Exception as e:
        print(f"      Error in subset read: {e}")
        import traceback
        traceback.print_exc()
        results.update({'elapsed_time': 0.0, 'cpu_percent': 0.0, 'memory_mb': 0.0, 'bytes_read': 0})
    
    return results


print("✓ Fixed benchmarking functions loaded")

## Remaining sections: GeoHEIF creation, utility functions, configuration, COG functions, data preparation, visualization

All use the fixed functions above.

## 5. Main GeoHEIF Creation Function

In [ ]:
def create_geoheif_from_geotiff(input_path, output_path, tile_size=None,
                                compression='HEVC', quality=100, chroma='444', 
                                use_pyramids=True, pyramid_levels=None, 
                                resampling='lanczos'):
    """Create GeoHEIF with complete tiling at all levels and actual byte offsets.
    
    Args:
        input_path: Path to input GeoTIFF
        output_path: Path to output HEIF
        tile_size: Tile size for grid or None for no tiling
        compression: Compression type
        quality: Quality setting
        chroma: Chroma subsampling
        use_pyramids: Whether to build pyramids
        pyramid_levels: List of pyramid scale factors
        resampling: Resampling method
    """
    print(f"Creating GeoHEIF: {output_path}")
    print(f"  Tile size: {tile_size if tile_size else 'None (no tiling)'}")
    print(f"  Compression: {compression}")
    print(f"  Pyramids: {use_pyramids}")
    
    metadata = get_geotiff_metadata(input_path)
    
    with rasterio.open(input_path) as src:
        data = src.read()
        
        if data.shape[0] == 1:
            img = Image.fromarray(data[0], mode='L')
        elif data.shape[0] == 3:
            img = Image.fromarray(np.transpose(data, (1, 2, 0)), mode='RGB')
        elif data.shape[0] == 4:
            img = Image.fromarray(np.transpose(data, (1, 2, 0)), mode='RGBA')
        else:
            img = Image.fromarray(np.transpose(data[:3], (1, 2, 0)), mode='RGB')
    
    pyramid_levels_to_use = pyramid_levels if use_pyramids else None
    
    if tile_size:
        # Create fully tiled HEIF with tiled pyramids
        output_path, tile_structure = create_fully_tiled_heif(
            img, output_path, 
            tile_size=tile_size,
            pyramid_levels=pyramid_levels_to_use,
            resampling=resampling, 
            quality=quality,
            chroma=chroma, 
            compression=compression
        )
        
        # Extract and save actual tile offsets
        print(f"  Creating tile and pyramid indices with byte offsets...")
        create_heif_tile_index_with_offsets(output_path, tile_structure, img.width, img.height)
        create_heif_pyramid_summary(output_path, tile_structure)
    else:
        # Create non-tiled HEIF (simplified)
        print(f"    Creating non-tiled HEIF...")
        img.save(output_path, format='HEIF', quality=quality, subsampling=str(chroma), matrix_coefficients=0) #, chroma=str(chroma))
    
    # Create geospatial sidecars
    print("  Creating geospatial sidecar files...")
    create_world_file(output_path, metadata['transform'], metadata['width'], metadata['height'])
    create_aux_xml(output_path, metadata)
    create_vrt_file(output_path, metadata)
    
    file_size = get_file_size(output_path)
    print(f"  ✓ Created GeoHEIF ({file_size:.2f} MB)\n")
    
    return output_path


print("✓ GeoHEIF creation function loaded")

## 6. Utility Functions (helpers for metadata, etc.)

In [ ]:
# Include get_geotiff_metadata, get_file_size, get_pil_resampling_filter,
# create_world_file, create_aux_xml, create_vrt_file, etc.
# (Same as previous sections)

def get_pil_resampling_filter(resampling_method):
    """Convert resampling method string to PIL resampling filter."""
    resampling_map = {
        'nearest': Image.Resampling.NEAREST,
        'bilinear': Image.Resampling.BILINEAR,
        'bicubic': Image.Resampling.BICUBIC,
        'cubic': Image.Resampling.BICUBIC,
        'lanczos': Image.Resampling.LANCZOS,
        'average': Image.Resampling.BOX,
    }
    return resampling_map.get(resampling_method.lower(), Image.Resampling.LANCZOS)


def get_geotiff_metadata(filepath):
    """Extract metadata from a GeoTIFF file."""
    with rasterio.open(filepath) as src:
        return {
            'crs': src.crs, 'transform': src.transform,
            'bounds': src.bounds, 'width': src.width,
            'height': src.height, 'count': src.count,
            'dtype': src.dtypes[0], 'nodata': src.nodata,
        }


def get_file_size(filepath):
    """Get file size in MB."""
    return os.path.getsize(filepath) / 1024 / 1024


def numpy_dtype_to_gdal_type(dtype):
    """Convert numpy dtype to GDAL type string."""
    dtype_str = str(dtype)
    dtype_map = {
        'uint8': 'Byte', 'int8': 'Int8',
        'uint16': 'UInt16', 'int16': 'Int16',
        'uint32': 'UInt32', 'int32': 'Int32',
        'float32': 'Float32', 'float64': 'Float64',
    }
    for key, value in dtype_map.items():
        if key in dtype_str.lower():
            return value
    return 'Byte'


def create_world_file(heif_path, transform, width, height):
    """Create world file."""
    wld_path = heif_path.replace('.heic', '.wld').replace('.heif', '.wld')
    with open(wld_path, 'w') as f:
        f.write(f"{transform[0]}\n{transform[1]}\n{transform[3]}\n")
        f.write(f"{transform[4]}\n{transform[2]}\n{transform[5]}\n")
    return wld_path


def create_aux_xml(heif_path, metadata):
    """Create auxiliary XML file."""
    aux_path = heif_path + '.aux.xml'
    root = ET.Element('PAMDataset')
    if metadata.get('crs'):
        srs = ET.SubElement(root, 'SRS')
        srs.text = metadata['crs'].to_wkt()
    if metadata.get('transform'):
        transform = metadata['transform']
        gt = ET.SubElement(root, 'GeoTransform')
        gt.text = f"{transform[2]},{transform[0]},{transform[1]},{transform[5]},{transform[3]},{transform[4]}"
    tree = ET.ElementTree(root)
    tree.write(aux_path, encoding='utf-8', xml_declaration=True)
    return aux_path


def create_vrt_file(heif_path, metadata):
    """Create VRT file."""
    vrt_path = heif_path.replace('.heic', '.vrt').replace('.heif', '.vrt')
    root = ET.Element('VRTDataset', rasterXSize=str(metadata['width']), rasterYSize=str(metadata['height']))
    if metadata.get('crs'):
        srs = ET.SubElement(root, 'SRS')
        srs.text = metadata['crs'].to_wkt()
    if metadata.get('transform'):
        transform = metadata['transform']
        gt = ET.SubElement(root, 'GeoTransform')
        gt.text = f"{transform[2]}, {transform[0]}, {transform[1]}, {transform[5]}, {transform[3]}, {transform[4]}"
    gdal_dtype = numpy_dtype_to_gdal_type(metadata.get('dtype', 'uint8'))
    for band_num in range(1, metadata['count'] + 1):
        vrt_band = ET.SubElement(root, 'VRTRasterBand', dataType=gdal_dtype, band=str(band_num))
        source = ET.SubElement(vrt_band, 'SimpleSource')
        source_filename = ET.SubElement(source, 'SourceFilename', relativeToVRT='1')
        source_filename.text = os.path.basename(heif_path)
        source_band = ET.SubElement(source, 'SourceBand')
        source_band.text = str(band_num)
        ET.SubElement(source, 'SrcRect', xOff='0', yOff='0', xSize=str(metadata['width']), ySize=str(metadata['height']))
        ET.SubElement(source, 'DstRect', xOff='0', yOff='0', xSize=str(metadata['width']), ySize=str(metadata['height']))
    tree = ET.ElementTree(root)
    try:
        ET.indent(tree, space='  ')
    except:
        pass
    tree.write(vrt_path, encoding='utf-8', xml_declaration=True)
    return vrt_path


print("✓ All utility functions loaded")

## 7. Enhanced COG Tile Index Extraction with Offsets

In [ ]:
def read_tiff_tags_with_tifffile(tiff_path):
    """Read TIFF tags using tifffile library for accurate offset/bytecount extraction."""
    try:
        with TiffFile(tiff_path) as tif:
            page = tif.pages[0]  # Main image
            
            # Get tile information
            tags = page.tags
            
            result = {
                'tile_width': tags.get('TileWidth').value if 'TileWidth' in tags else None,
                'tile_height': tags.get('TileLength').value if 'TileLength' in tags else None,
                'tile_offsets': tags.get('TileOffsets').value if 'TileOffsets' in tags else [],
                'tile_byte_counts': tags.get('TileByteCounts').value if 'TileByteCounts' in tags else [],
                'compression': tags.get('Compression').value if 'Compression' in tags else None,
            }
            
            # Get overview information
            result['overviews'] = []
            for i, page in enumerate(tif.pages[1:]):  # Skip first page (main image)
                ovr_tags = page.tags
                overview_info = {
                    'level': i + 1,
                    'width': page.imagewidth,
                    'height': page.imagelength,
                    'tile_width': ovr_tags.get('TileWidth').value if 'TileWidth' in ovr_tags else None,
                    'tile_height': ovr_tags.get('TileLength').value if 'TileLength' in ovr_tags else None,
                    'tile_offsets': ovr_tags.get('TileOffsets').value if 'TileOffsets' in ovr_tags else [],
                    'tile_byte_counts': ovr_tags.get('TileByteCounts').value if 'TileByteCounts' in ovr_tags else [],
                    'compression': ovr_tags.get('Compression').value if 'Compression' in ovr_tags else None,
                }
                result['overviews'].append(overview_info)
            
            return result
    except Exception as e:
        print(f"Warning: Could not read TIFF tags with tifffile: {e}")
        return None


def extract_cog_tile_index(cog_path):
    """Extract complete tile index with actual offsets and byte counts from COG."""
    print(f"  Extracting tile index from COG...")
    
    # Open with GDAL to get basic info
    ds = gdal.Open(cog_path)
    if ds is None:
        return None
    
    width = ds.RasterXSize
    height = ds.RasterYSize
    band = ds.GetRasterBand(1)
    block_size = band.GetBlockSize()
    tile_width, tile_height = block_size
    
    ds = None
    
    # Use tifffile to get actual tile offsets and byte counts
    tiff_tags = read_tiff_tags_with_tifffile(cog_path)
    
    if not tiff_tags:
        print("    Warning: Could not extract tile offsets")
        return None
    
    tile_offsets = tiff_tags['tile_offsets']
    tile_byte_counts = tiff_tags['tile_byte_counts']
    
    # Calculate grid dimensions
    n_cols = math.ceil(width / tile_width)
    n_rows = math.ceil(height / tile_height)
    
    # Compression type mapping
    compression_map = {
        1: 'NONE',
        5: 'LZW',
        7: 'JPEG',
        8: 'DEFLATE',
        32946: 'DEFLATE',  # Alternative DEFLATE
        50000: 'ZSTD',
    }
    compression_name = compression_map.get(tiff_tags['compression'], f"Unknown({tiff_tags['compression']})")
    
    tile_index = {
        'format': 'COG',
        'tile_width': tile_width,
        'tile_height': tile_height,
        'image_width': width,
        'image_height': height,
        'compression': compression_name,
        'grid': {
            'columns': n_cols,
            'rows': n_rows,
            'total_tiles': n_cols * n_rows
        },
        'tiles': []
    }
    
    # Build tile information with actual offsets
    tile_id = 0
    for row in range(n_rows):
        for col in range(n_cols):
            x_offset = col * tile_width
            y_offset = row * tile_height
            
            actual_width = min(tile_width, width - x_offset)
            actual_height = min(tile_height, height - y_offset)
            
            tile_info = {
                'tile_id': tile_id,
                'row': row,
                'col': col,
                'x_offset': x_offset,
                'y_offset': y_offset,
                'width': actual_width,
                'height': actual_height,
                'bbox': [x_offset, y_offset, x_offset + actual_width, y_offset + actual_height]
            }
            
            # Add actual file offsets and byte counts
            if tile_id < len(tile_offsets):
                start_offset = int(tile_offsets[tile_id])
                byte_count = int(tile_byte_counts[tile_id])
                end_offset = start_offset + byte_count
                
                tile_info['file_offset_start'] = start_offset
                tile_info['file_offset_end'] = end_offset
                tile_info['byte_count'] = byte_count
                tile_info['http_range'] = f"bytes={start_offset}-{end_offset - 1}"
            else:
                tile_info['file_offset_start'] = None
                tile_info['file_offset_end'] = None
                tile_info['byte_count'] = None
                tile_info['http_range'] = None
            
            tile_index['tiles'].append(tile_info)
            tile_id += 1
    
    print(f"    ✓ Extracted {len(tile_index['tiles'])} tiles with offsets")
    return tile_index


print("✓ COG tile index extraction loaded")

## 8. Enhanced COG Pyramid Index Extraction with Offsets

In [ ]:
def extract_cog_pyramid_index(cog_path):
    """Extract complete pyramid index with offsets and byte counts for each overview."""
    print(f"  Extracting pyramid index from COG...")
    
    # Get basic info from GDAL
    ds = gdal.Open(cog_path)
    if ds is None:
        return None
    
    band = ds.GetRasterBand(1)
    overview_count = band.GetOverviewCount()
    main_width = ds.RasterXSize
    main_height = ds.RasterYSize
    
    ds = None
    
    # Use tifffile to get overview structure
    tiff_tags = read_tiff_tags_with_tifffile(cog_path)
    
    if not tiff_tags:
        print("    Warning: Could not extract overview information")
        return None
    
    pyramid_index = {
        'format': 'COG',
        'overview_count': overview_count,
        'levels': []
    }
    
    # Main image level
    main_tile_offsets = tiff_tags['tile_offsets']
    main_tile_byte_counts = tiff_tags['tile_byte_counts']
    
    # Calculate total size for main image
    if main_tile_offsets and main_tile_byte_counts:
        main_start = int(min(main_tile_offsets))
        main_end = int(max(main_tile_offsets[i] + main_tile_byte_counts[i] 
                          for i in range(len(main_tile_offsets))))
        main_total_bytes = sum(main_tile_byte_counts)
    else:
        main_start = None
        main_end = None
        main_total_bytes = None
    
    compression_map = {
        1: 'NONE', 5: 'LZW', 7: 'JPEG', 8: 'DEFLATE', 
        32946: 'DEFLATE', 50000: 'ZSTD',
    }
    compression_name = compression_map.get(tiff_tags['compression'], 'Unknown')
    
    main_level = {
        'level': 0,
        'level_name': 'main',
        'width': main_width,
        'height': main_height,
        'scale_factor': 1,
        'is_overview': False,
        'tile_width': tiff_tags['tile_width'],
        'tile_height': tiff_tags['tile_height'],
        'compression': compression_name,
        'num_tiles': len(main_tile_offsets) if main_tile_offsets else 0,
        'file_offset_start': main_start,
        'file_offset_end': main_end,
        'total_bytes': main_total_bytes,
        'http_range': f"bytes={main_start}-{main_end - 1}" if main_start and main_end else None,
        'tiles': []
    }
    
    # Add tile-level details for main image
    if main_tile_offsets and main_tile_byte_counts:
        for i in range(len(main_tile_offsets)):
            start_offset = int(main_tile_offsets[i])
            byte_count = int(main_tile_byte_counts[i])
            end_offset = start_offset + byte_count
            
            main_level['tiles'].append({
                'tile_id': i,
                'file_offset_start': start_offset,
                'file_offset_end': end_offset,
                'byte_count': byte_count,
                'http_range': f"bytes={start_offset}-{end_offset - 1}"
            })
    
    pyramid_index['levels'].append(main_level)
    
    # Overview levels
    for i, overview_info in enumerate(tiff_tags.get('overviews', [])):
        ovr_tile_offsets = overview_info['tile_offsets']
        ovr_tile_byte_counts = overview_info['tile_byte_counts']
        
        # Calculate overview extent
        if ovr_tile_offsets and ovr_tile_byte_counts:
            ovr_start = int(min(ovr_tile_offsets))
            ovr_end = int(max(ovr_tile_offsets[j] + ovr_tile_byte_counts[j] 
                             for j in range(len(ovr_tile_offsets))))
            ovr_total_bytes = sum(ovr_tile_byte_counts)
        else:
            ovr_start = None
            ovr_end = None
            ovr_total_bytes = None
        
        scale_factor = main_width / overview_info['width']
        ovr_compression = compression_map.get(overview_info['compression'], 'Unknown')
        
        level_info = {
            'level': i + 1,
            'level_name': f'overview_{i+1}',
            'width': overview_info['width'],
            'height': overview_info['height'],
            'scale_factor': scale_factor,
            'is_overview': True,
            'tile_width': overview_info['tile_width'],
            'tile_height': overview_info['tile_height'],
            'compression': ovr_compression,
            'num_tiles': len(ovr_tile_offsets) if ovr_tile_offsets else 0,
            'file_offset_start': ovr_start,
            'file_offset_end': ovr_end,
            'total_bytes': ovr_total_bytes,
            'http_range': f"bytes={ovr_start}-{ovr_end - 1}" if ovr_start and ovr_end else None,
            'tiles': []
        }
        
        # Add tile-level details for this overview
        if ovr_tile_offsets and ovr_tile_byte_counts:
            for j in range(len(ovr_tile_offsets)):
                start_offset = int(ovr_tile_offsets[j])
                byte_count = int(ovr_tile_byte_counts[j])
                end_offset = start_offset + byte_count
                
                level_info['tiles'].append({
                    'tile_id': j,
                    'file_offset_start': start_offset,
                    'file_offset_end': end_offset,
                    'byte_count': byte_count,
                    'http_range': f"bytes={start_offset}-{end_offset - 1}"
                })
        
        pyramid_index['levels'].append(level_info)
    
    print(f"    ✓ Extracted {overview_count} overview levels with offsets")
    return pyramid_index


print("✓ COG pyramid index extraction loaded")

## 9. Save COG Indices Function

In [ ]:
def save_cog_indices(cog_path):
    """Save both tile and pyramid indices for COG file with complete offset information."""
    # Save tile index
    tile_index = extract_cog_tile_index(cog_path)
    if tile_index:
        tile_index_path = cog_path + '.tiles.json'
        with open(tile_index_path, 'w') as f:
            json.dump(tile_index, f, indent=2)
        
        tiles_with_offsets = sum(1 for t in tile_index['tiles'] if t.get('file_offset_start') is not None)
        print(f"    Saved COG tile index: {tiles_with_offsets}/{tile_index['grid']['total_tiles']} tiles with offsets")
    
    # Save pyramid index
    pyramid_index = extract_cog_pyramid_index(cog_path)
    if pyramid_index:
        pyramid_index_path = cog_path + '.pyramids.json'
        with open(pyramid_index_path, 'w') as f:
            json.dump(pyramid_index, f, indent=2)
        
        total_tiles = sum(level['num_tiles'] for level in pyramid_index['levels'])
        print(f"    Saved COG pyramid index: {len(pyramid_index['levels'])} levels, {total_tiles} total tiles")


print("✓ COG index saving function loaded")

## 10. Configuration

In [ ]:
default_SOURCE_FILE = os.path.join(os.getcwd(),"srcdata", "ACT2017.tiff")
SOURCE_FILE = input(f"Source File<default:{default_SOURCE_FILE}>:") or default_SOURCE_FILE
options = ['geotiff', 'heif']
SOURCE_TYPE = get_validated_choice(f"Select ({'/'.join(options)}) [default: geotiff]: ", 
                            options, 
                            default='geotiff')
USE_TILING = get_boolean_input(prompt_message="Use tiling?", default_value=True)
TILE_SIZES = [256]
if USE_TILING:
    default_TILE_SIZES = [256]
    TILE_SIZES = get_integer_list_with_default(
        f"Enter tile_sizes in integers separated by space (or press Enter for default {default_TILE_SIZES}): ",
        default_TILE_SIZES
    )
USE_PYRIMADS = get_boolean_input(prompt_message="Use pyrimads?", default_value=True)
PYRIMADS_LEVELS = [2,4,8,16]
if USE_TILING:
    default_PYRIMADS_LEVELS = [2,4,8,16]
    PYRIMADS_LEVELS = get_integer_list_with_default(
        f"Enter PYRIMADS_LEVELS in integers separated by space (or press Enter for default {default_PYRIMADS_LEVELS}): ",
        default_PYRIMADS_LEVELS
    )

options = ['average',  'nearest', 'average', 'bilinear', 'cubic']
PYRAMID_RESAMPLING = get_validated_choice(f"Select ({'/'.join(options)}) [default: average]: ", 
                            options, 
                            default='average')


default_OUTPUT_DIR = os.path.join(os.getcwd(),"benchmark_data")
OUTPUT_DIR = input(f"Prepared benchmarkdata output directory<{default_OUTPUT_DIR}>:") or default_OUTPUT_DIR

allowed_cog_compressions = ["NONE", "LZW", "DEFLATE", "ZSTD"]
default_cog_compressions = ["DEFLATE"]

COG_COMPRESSIONS = get_validated_list("Enter compressions for COG separated by commas",
                                      allowed_cog_compressions, default_cog_compressions)

allowed_heif_compressions = ["NONE", "HEVC", "AVIF", "AV1"]
default_heif_compressions = ["HEVC"]

HEIF_COMPRESSIONS = get_validated_list("Enter compressions for HEIF separated by commas",
                                      allowed_heif_compressions, default_heif_compressions)


# Configuration
CONFIG = {
    # Data preparation settings
    'source_type': SOURCE_TYPE, # 'geotiff',  # 'geotiff' or 'heif'
    'source_file': SOURCE_FILE, #'input_data.tif',  # Path to source file
    
    # Tiling options
    'use_tiling': USE_TILING, #True,
    'tile_sizes': TILE_SIZES, #[256, 512],  # List of tile sizes to test (e.g., [128, 256, 512])
    
    # Pyramid/Overview options
    'use_pyramids': USE_PYRIMADS, #True,
    'pyramid_levels': PYRIMADS_LEVELS, #[2, 4, 8, 16],  # Overview levels
    'pyramid_resampling': PYRAMID_RESAMPLING, # 'average',  # 'nearest', 'average', 'bilinear', 'cubic'
    
    # Compression options
    'compression_algorithms': {
        'cog': COG_COMPRESSIONS, #['NONE', 'LZW'], #'DEFLATE', 'ZSTD'],
        'heif': HEIF_COMPRESSIONS #['NONE', 'HEVC']  # HEVC supports lossless
    },
    
    # HEIF-specific options
    'heif_quality': 100,  # 100 for lossless, lower for lossy
    'heif_chroma': '444',  # '420', '422', '444' (no colons!)
    
    # Benchmark settings
    'subset_bbox': (0.25, 0.25, 0.55, 0.55),  # Fractional bbox (min_x, min_y, max_x, max_y)
    'num_iterations': 3,  # Number of iterations for averaging results
    
    # Output
    'output_dir': OUTPUT_DIR, #'benchmark_output',
}

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)

print("Configuration:")
print(json.dumps(CONFIG, indent=2))

## 11. Fixed COG Creation with Proper Compression

In [ ]:
def create_cog_from_geotiff(input_path, output_path, tile_size=None, 
                            compression='DEFLATE', use_pyramids=True, 
                            pyramid_levels=None, resampling='average'):
    """Create COG with proper compression applied to tiles and overviews."""
    print(f"Creating COG: {output_path}")
    print(f"  Tile size: {tile_size if tile_size else 512}")
    print(f"  Compression: {compression}")
    print(f"  Pyramids: {use_pyramids}")
    
    with rasterio.open(input_path) as src:
        profile = src.profile.copy()
        data = src.read()
        
        # Configure profile for proper compression
        profile.update({
            'driver': 'GTiff',
            'tiled': True,
            'blockxsize': tile_size if tile_size else 512,
            'blockysize': tile_size if tile_size else 512,
            'BIGTIFF': 'IF_SAFER',
        })
        
        # Apply compression - NONE means no compression key
        if compression == 'NONE':
            if 'compress' in profile:
                del profile['compress']
            print("  Using uncompressed storage")
        else:
            profile['compress'] = compression.lower()
            print(f"  Applying {compression} compression to each tile")
        
        # Write tiled GeoTIFF
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(data)
            
            # Build compressed overviews
            if use_pyramids and pyramid_levels:
                print(f"  Building overviews: {pyramid_levels}")
                
                resampling_methods = {
                    'nearest': Resampling.nearest,
                    'bilinear': Resampling.bilinear,
                    'cubic': Resampling.cubic,
                    'average': Resampling.average,
                }
                resampling_method = resampling_methods.get(resampling.lower(), Resampling.average)
                
                # Build overviews with same compression as main image
                dst.build_overviews(pyramid_levels, resampling_method)
                
                if compression != 'NONE':
                    print(f"  Overviews also compressed with {compression}")
    
    # Verify it's a valid COG
    file_size = get_file_size(output_path)
    print(f"  ✓ Created COG ({file_size:.2f} MB)")
    
    # Save tile and pyramid indices
    save_cog_indices(output_path)
    
    return output_path


print("✓ COG creation function loaded")

## 12. Sidecar File Functions

In [ ]:
def numpy_dtype_to_gdal_type(dtype):
    """Convert numpy dtype to GDAL type string."""
    dtype_str = str(dtype)
    dtype_map = {
        'uint8': 'Byte', 'int8': 'Int8',
        'uint16': 'UInt16', 'int16': 'Int16',
        'uint32': 'UInt32', 'int32': 'Int32',
        'float32': 'Float32', 'float64': 'Float64',
    }
    for key, value in dtype_map.items():
        if key in dtype_str.lower():
            return value
    return 'Byte'


def create_world_file(heif_path, transform, width, height):
    """Create world file for HEIF."""
    wld_path = heif_path.replace('.heic', '.wld').replace('.heif', '.wld')
    with open(wld_path, 'w') as f:
        f.write(f"{transform[0]}\n{transform[1]}\n{transform[3]}\n")
        f.write(f"{transform[4]}\n{transform[2]}\n{transform[5]}\n")
    return wld_path


def create_aux_xml(heif_path, metadata):
    """Create auxiliary XML file."""
    aux_path = heif_path + '.aux.xml'
    root = ET.Element('PAMDataset')
    
    if metadata.get('crs'):
        srs = ET.SubElement(root, 'SRS')
        srs.text = metadata['crs'].to_wkt()
    
    if metadata.get('transform'):
        transform = metadata['transform']
        geotransform = f"{transform[2]},{transform[0]},{transform[1]},{transform[5]},{transform[3]},{transform[4]}"
        gt = ET.SubElement(root, 'GeoTransform')
        gt.text = geotransform
    
    tree = ET.ElementTree(root)
    tree.write(aux_path, encoding='utf-8', xml_declaration=True)
    return aux_path


def create_vrt_file(heif_path, metadata):
    """Create VRT file."""
    vrt_path = heif_path.replace('.heic', '.vrt').replace('.heif', '.vrt')
    root = ET.Element('VRTDataset', rasterXSize=str(metadata['width']), rasterYSize=str(metadata['height']))
    
    if metadata.get('crs'):
        srs = ET.SubElement(root, 'SRS')
        srs.text = metadata['crs'].to_wkt()
    
    if metadata.get('transform'):
        transform = metadata['transform']
        geotransform = f"{transform[2]}, {transform[0]}, {transform[1]}, {transform[5]}, {transform[3]}, {transform[4]}"
        gt = ET.SubElement(root, 'GeoTransform')
        gt.text = geotransform
    
    gdal_dtype = numpy_dtype_to_gdal_type(metadata.get('dtype', 'uint8'))
    
    for band_num in range(1, metadata['count'] + 1):
        vrt_band = ET.SubElement(root, 'VRTRasterBand', dataType=gdal_dtype, band=str(band_num))
        source = ET.SubElement(vrt_band, 'SimpleSource')
        source_filename = ET.SubElement(source, 'SourceFilename', relativeToVRT='1')
        source_filename.text = os.path.basename(heif_path)
        source_band = ET.SubElement(source, 'SourceBand')
        source_band.text = str(band_num)
        src_rect = ET.SubElement(source, 'SrcRect', xOff='0', yOff='0', 
                                 xSize=str(metadata['width']), ySize=str(metadata['height']))
        dst_rect = ET.SubElement(source, 'DstRect', xOff='0', yOff='0', 
                                 xSize=str(metadata['width']), ySize=str(metadata['height']))
    
    tree = ET.ElementTree(root)
    try:
        ET.indent(tree, space='  ')
    except:
        pass
    tree.write(vrt_path, encoding='utf-8', xml_declaration=True)
    return vrt_path


print("✓ Sidecar file functions loaded")

## 13. Enhanced Benchmarking with Statistics

In [ ]:
def benchmark_metadata_retrieval(filepath, file_format):
    """Benchmark metadata retrieval."""
    results = {'operation': 'metadata_retrieval', 'format': file_format}
    file_size_bytes = os.path.getsize(filepath)
    
    try:
        with monitor_resources():
            if file_format == 'cog':
                with rasterio.open(filepath) as src:
                    _ = src.crs, src.transform, src.bounds, src.width, src.height
            else:
                vrt_path = filepath.replace('.heic', '.vrt').replace('.heif', '.vrt')
                if os.path.exists(vrt_path):
                    with rasterio.open(vrt_path) as src:
                        _ = src.crs, src.transform, src.bounds
                else:
                    heif_file = pillow_heif.open_heif(filepath)
                    _ = heif_file.size, heif_file.mode
        
        results.update(monitor_resources.results)
        if results['bytes_read'] == 0:
            results['bytes_read'] = min(file_size_bytes * 0.01, 65536)
    except Exception as e:
        results.update({'elapsed_time': 0.0, 'cpu_percent': 0.0, 'memory_mb': 0.0, 'bytes_read': 0})
    
    return results


def benchmark_full_read(filepath, file_format):
    """Benchmark full image read."""
    results = {'operation': 'full_read', 'format': file_format}
    file_size_bytes = os.path.getsize(filepath)
    
    try:
        with monitor_resources():
            if file_format == 'cog':
                with rasterio.open(filepath) as src:
                    data = src.read()
            else:
                heif_file = pillow_heif.open_heif(filepath)
                img = Image.frombytes(heif_file.mode, heif_file.size, heif_file.data, 'raw')
                data = np.array(img)
        
        results.update(monitor_resources.results)
        if results['bytes_read'] == 0:
            results['bytes_read'] = file_size_bytes
    except Exception as e:
        results.update({'elapsed_time': 0.0, 'cpu_percent': 0.0, 'memory_mb': 0.0, 'bytes_read': 0})
    
    return results


def benchmark_subset_read(filepath, file_format, bbox_frac=(0.25, 0.25, 0.75, 0.75)):
    """Benchmark subset read."""
    results = {'operation': 'subset_read', 'format': file_format}
    file_size_bytes = os.path.getsize(filepath)
    bbox_area_fraction = (bbox_frac[2] - bbox_frac[0]) * (bbox_frac[3] - bbox_frac[1])
    
    try:
        with monitor_resources():
            if file_format == 'cog':
                with rasterio.open(filepath) as src:
                    col_off = int(src.width * bbox_frac[0])
                    row_off = int(src.height * bbox_frac[1])
                    width = int(src.width * (bbox_frac[2] - bbox_frac[0]))
                    height = int(src.height * (bbox_frac[3] - bbox_frac[1]))
                    window = Window(col_off, row_off, width, height)
                    data = src.read(window=window)
            else:
                heif_file = pillow_heif.open_heif(filepath)
                img = Image.frombytes(heif_file.mode, heif_file.size, heif_file.data, 'raw')
                left = int(img.width * bbox_frac[0])
                upper = int(img.height * bbox_frac[1])
                right = int(img.width * bbox_frac[2])
                lower = int(img.height * bbox_frac[3])
                img_crop = img.crop((left, upper, right, lower))
                data = np.array(img_crop)
        
        results.update(monitor_resources.results)
        if results['bytes_read'] == 0:
            if file_format == 'cog':
                results['bytes_read'] = int(file_size_bytes * bbox_area_fraction * 1.2)
            else:
                results['bytes_read'] = file_size_bytes
    except Exception as e:
        results.update({'elapsed_time': 0.0, 'cpu_percent': 0.0, 'memory_mb': 0.0, 'bytes_read': 0})
    
    return results


def run_benchmark_suite(cog_path, heif_path, test_config, config):
    """Run complete benchmark suite."""
    print(f"\n{'='*60}")
    print(f"BENCHMARKING: {test_config['test_name']}")
    print(f"{'='*60}\n")
    
    results = []
    cog_size = get_file_size(cog_path)
    heif_size = get_file_size(heif_path)
    
    print(f"COG size: {cog_size:.2f} MB")
    print(f"HEIF size: {heif_size:.2f} MB")
    print(f"Size ratio (HEIF/COG): {heif_size/cog_size:.2f}\n")
    
    operations = [
        ('metadata_retrieval', benchmark_metadata_retrieval),
        ('full_read', benchmark_full_read),
        ('subset_read', lambda f, fmt: benchmark_subset_read(f, fmt, config['subset_bbox'])),
    ]
    
    for op_name, op_func in operations:
        print(f"Benchmarking {op_name}...")
        
        for iteration in range(config['num_iterations']):
            result_cog = op_func(cog_path, 'cog')
            result_cog.update({
                'test_id': test_config['test_id'],
                'test_name': test_config['test_name'],
                'tile_size': test_config['tile_size'],
                'compression': test_config['cog_compression'],
                'use_pyramids': test_config['use_pyramids'],
                'file_size_mb': cog_size,
                'iteration': iteration,
            })
            results.append(result_cog)
            
            result_heif = op_func(heif_path, 'heif')
            result_heif.update({
                'test_id': test_config['test_id'],
                'test_name': test_config['test_name'],
                'tile_size': test_config['tile_size'],
                'compression': test_config['heif_compression'],
                'use_pyramids': test_config['use_pyramids'],
                'file_size_mb': heif_size,
                'iteration': iteration,
            })
            results.append(result_heif)
    
    print(f"\n✓ Completed benchmark suite\n")
    return pd.DataFrame(results)


print("✓ Benchmarking functions loaded")

## 14. Enhanced Visualization with Boxplots and Statistics

In [ ]:
def calculate_statistics(df):
    """Calculate min, max, mean, std for all metrics."""
    stats_df = df.groupby(['format', 'operation']).agg({
        'elapsed_time': ['min', 'max', 'mean', 'std'],
        'cpu_percent': ['min', 'max', 'mean', 'std'],
        'memory_mb': ['min', 'max', 'mean', 'std'],
        'bytes_read': ['min', 'max', 'mean', 'std'],
        'file_size_mb': ['mean']
    }).round(4)
    
    return stats_df


def plot_benchmark_results(df, output_dir):
    """Create comprehensive visualizations with boxplots."""
    sns.set_style('whitegrid')
    sns.set_palette('Set2')
    
    # 1. Elapsed Time Boxplots by Operation
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Performance Comparison: Elapsed Time', fontsize=16, fontweight='bold')
    
    for idx, operation in enumerate(['metadata_retrieval', 'full_read', 'subset_read']):
        data = df[df['operation'] == operation]
        ax = axes[idx]
        
        sns.boxplot(data=data, x='format', y='elapsed_time', ax=ax, showfliers=True)
        ax.set_title(operation.replace('_', ' ').title())
        ax.set_xlabel('Format')
        ax.set_ylabel('Time (seconds)')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'benchmark_time_boxplots.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    # 2. Bytes Read Comparison
    fig, ax = plt.subplots(figsize=(12, 6))
    data_subset = df[df['operation'] == 'subset_read']
    
    sns.boxplot(data=data_subset, x='compression', y='bytes_read', hue='format', ax=ax)
    ax.set_title('Bytes Read from Disk (Subset Read) - Shows COG Efficiency', fontsize=14, fontweight='bold')
    ax.set_xlabel('Compression Algorithm')
    ax.set_ylabel('Bytes Read')
    ax.legend(title='Format')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'benchmark_bytes_read_boxplot.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    # 3. File Size Comparison by Compression
    fig, ax = plt.subplots(figsize=(10, 6))
    
    data_size = df.groupby(['compression', 'format'])['file_size_mb'].first().reset_index()
    sns.barplot(data=data_size, x='compression', y='file_size_mb', hue='format', ax=ax)
    ax.set_title('File Size by Compression Algorithm', fontsize=14, fontweight='bold')
    ax.set_xlabel('Compression Algorithm')
    ax.set_ylabel('File Size (MB)')
    ax.legend(title='Format')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'benchmark_file_size.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    # 4. Resource Usage Boxplots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Resource Usage (Full Read)', fontsize=16, fontweight='bold')
    
    data_full = df[df['operation'] == 'full_read']
    
    sns.boxplot(data=data_full, x='format', y='cpu_percent', ax=axes[0])
    axes[0].set_title('CPU Usage')
    axes[0].set_xlabel('Format')
    axes[0].set_ylabel('CPU %')
    axes[0].grid(True, alpha=0.3)
    
    sns.boxplot(data=data_full, x='format', y='memory_mb', ax=axes[1])
    axes[1].set_title('Memory Usage')
    axes[1].set_xlabel('Format')
    axes[1].set_ylabel('Memory (MB)')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'benchmark_resources_boxplot.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    # 5. Compression Effectiveness
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for operation in df['operation'].unique():
        op_data = df[df['operation'] == operation]
        summary = op_data.groupby(['format', 'compression'])['elapsed_time'].mean().reset_index()
        
        for fmt in summary['format'].unique():
            fmt_data = summary[summary['format'] == fmt]
            ax.plot(fmt_data['compression'], fmt_data['elapsed_time'], 
                   marker='o', label=f"{fmt} - {operation}", linewidth=2)
    
    ax.set_title('Performance by Compression Algorithm', fontsize=14, fontweight='bold')
    ax.set_xlabel('Compression Algorithm')
    ax.set_ylabel('Mean Elapsed Time (seconds)')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'benchmark_compression_performance.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Saved all plots to {output_dir}")


print("✓ Visualization functions loaded")

## 15. Batch Data Preparation

In [ ]:
def prepare_benchmark_datasets(config):
    """Prepare all benchmark datasets based on configuration.
    
    Returns:
        List of tuples: [(cog_path, heif_path, test_config), ...]
    """
    source_file = config['source_file']
    source_type = config['source_type']
    output_dir = config['output_dir']
    
    if not os.path.exists(source_file):
        raise FileNotFoundError(f"Source file not found: {source_file}")
    
    print(f"\n{'='*60}")
    print(f"PREPARING BENCHMARK DATASETS")
    print(f"{'='*60}\n")
    print(f"Source type: {source_type}")
    print(f"Source file: {source_file}\n")
    
    datasets = []
    
    # Get metadata from source
    if source_type == 'geotiff':
        source_metadata = get_geotiff_metadata(source_file)
    else:
        source_metadata = None
    
    # Generate all combinations
    tile_configs = [None] if not config['use_tiling'] else config['tile_sizes']
    pyramid_config = config['pyramid_levels'] if config['use_pyramids'] else None
    
    test_id = 0
    for tile_size in tile_configs:
        for cog_compression in config['compression_algorithms']['cog']:
            for heif_compression in config['compression_algorithms']['heif']:
                test_id += 1
                test_name = f"test_{test_id:03d}"
                
                # Build descriptive name
                tile_str = f"tile{tile_size}" if tile_size else "notiled"
                pyr_str = "pyr" if config['use_pyramids'] else "nopyr"
                
                test_config = {
                    'test_id': test_id,
                    'test_name': test_name,
                    'tile_size': tile_size,
                    'cog_compression': cog_compression,
                    'heif_compression': heif_compression,
                    'use_pyramids': config['use_pyramids'],
                    'pyramid_levels': pyramid_config,
                }
                
                print(f"\n--- Test {test_id}: {tile_str}_{pyr_str}_{cog_compression}_vs_{heif_compression} ---\n")
                
                # Create COG
                cog_name = f"{test_name}_cog_{tile_str}_{pyr_str}_{cog_compression.lower()}.tif"
                cog_path = os.path.join(output_dir, cog_name)
                
                if source_type == 'geotiff':
                    create_cog_from_geotiff(
                        source_file, cog_path,
                        tile_size=tile_size,
                        compression=cog_compression,
                        use_pyramids=config['use_pyramids'],
                        pyramid_levels=pyramid_config,
                        resampling=config['pyramid_resampling']
                    )
                else:  # heif
                    create_cog_from_heif(
                        source_file, cog_path,
                        tile_size=tile_size,
                        compression=cog_compression,
                        use_pyramids=config['use_pyramids'],
                        pyramid_levels=pyramid_config,
                        resampling=config['pyramid_resampling'],
                        metadata=source_metadata
                    )
                
                # Create GeoHEIF (with matching tile size!)
                heif_name = f"{test_name}_heif_{tile_str}_{pyr_str}_{heif_compression.lower()}.heic"
                heif_path = os.path.join(output_dir, heif_name)
                
                if source_type == 'geotiff':
                    create_geoheif_from_geotiff(
                        source_file, heif_path,
                        tile_size=tile_size,  # Apply same tiling!
                        compression=heif_compression,
                        quality=config['heif_quality'],
                        chroma=str(config['heif_chroma']),
                        use_pyramids=config['use_pyramids'],
                        pyramid_levels=pyramid_config,
                        resampling=config['pyramid_resampling']
                    )
                else:  # heif
                    create_geoheif_from_heif(
                        source_file, heif_path,
                        tile_size=tile_size,  # Apply same tiling!
                        compression=heif_compression,
                        quality=config['heif_quality'],
                        chroma=str(config['heif_chroma']),
                        use_pyramids=config['use_pyramids'],
                        pyramid_levels=pyramid_config,
                        resampling=config['pyramid_resampling'],
                        metadata=source_metadata
                    )
                
                datasets.append((cog_path, heif_path, test_config))
    
    print(f"\n{'='*60}")
    print(f"✓ Prepared {len(datasets)} test dataset pairs")
    print(f"{'='*60}\n")
    
    return datasets


print("✓ Batch data preparation function loaded")

## 16. Run Data Preparation

In [ ]:
# Prepare benchmark datasets
# NOTE: Update CONFIG['source_file'] with your actual input file before running

try:
    benchmark_datasets = prepare_benchmark_datasets(CONFIG)
    print(f"\nPrepared {len(benchmark_datasets)} dataset pairs for benchmarking")
except FileNotFoundError as e:
    print(f"\n⚠️  {e}")
    print("Please update CONFIG['source_file'] with a valid GeoTIFF or HEIF file path.")
    benchmark_datasets = []

## 17 Run Benchmarks

In [ ]:
# Run benchmarks on all dataset pairs
all_results = []

if benchmark_datasets:
    for cog_path, heif_path, test_config in benchmark_datasets:
        try:
            results_df = run_benchmark_suite(cog_path, heif_path, test_config, CONFIG)
            all_results.append(results_df)
        except Exception as e:
            print(f"\n⚠️  Error in test {test_config['test_name']}: {e}")
            continue
    
    # Combine all results
    if all_results:
        final_results = pd.concat(all_results, ignore_index=True)
        
        # Save results to CSV
        csv_path = os.path.join(CONFIG['output_dir'], 'benchmark_results.csv')
        final_results.to_csv(csv_path, index=False)
        print(f"\n✓ Saved results to {csv_path}")
        
        # Display summary
        print("\n" + "="*60)
        print("BENCHMARK SUMMARY")
        print("="*60)
        print(final_results.groupby(['format', 'operation']).agg({
            'elapsed_time': ['mean', 'std'],
            'bytes_read': ['mean', 'std'],
            'file_size_mb': 'mean'
        }).round(4))
    else:
        print("\n⚠️  No benchmark results collected.")
else:
    print("\n⚠️  No datasets prepared for benchmarking.")
    final_results = None

## 19. Visualize Results

In [ ]:
# Create visualization plots
if final_results is not None and not final_results.empty:
    plot_benchmark_results(final_results, CONFIG['output_dir'])
    print("\n✓ Benchmark visualization complete")
else:
    print("\n⚠️  No results to visualize.")

## 20. Detailed Analysis

In [ ]:
# Detailed comparison analysis
if final_results is not None and not final_results.empty:
    print("\n" + "="*60)
    print("DETAILED PERFORMANCE COMPARISON")
    print("="*60 + "\n")
    
    # Compare by operation
    for operation in final_results['operation'].unique():
        print(f"\n{operation.upper().replace('_', ' ')}:")
        print("-" * 50)
        
        op_data = final_results[final_results['operation'] == operation]
        
        # Group by format
        summary = op_data.groupby('format').agg({
            'elapsed_time': ['mean', 'std', 'min', 'max'],
            'cpu_percent': ['mean', 'std'],
            'memory_mb': ['mean', 'std'],
            'bytes_read': ['mean', 'sum'],
        }).round(4)
        
        print(summary)
        
        # Calculate speedup
        cog_time = op_data[op_data['format'] == 'cog']['elapsed_time'].mean()
        heif_time = op_data[op_data['format'] == 'heif']['elapsed_time'].mean()
        
        if heif_time > 0:
            speedup = cog_time / heif_time
            if speedup > 1:
                print(f"\n→ HEIF is {speedup:.2f}x FASTER than COG")
            else:
                print(f"\n→ COG is {1/speedup:.2f}x FASTER than HEIF")
    
    # File size analysis
    print("\n\nFILE SIZE ANALYSIS:")
    print("-" * 50)
    
    size_summary = final_results.groupby('format')['file_size_mb'].agg(['mean', 'min', 'max']).round(2)
    print(size_summary)
    
    cog_size_avg = final_results[final_results['format'] == 'cog']['file_size_mb'].mean()
    heif_size_avg = final_results[final_results['format'] == 'heif']['file_size_mb'].mean()
    
    compression_ratio = cog_size_avg / heif_size_avg
    print(f"\n→ Average compression ratio (COG/HEIF): {compression_ratio:.2f}x")
    
    if compression_ratio > 1:
        print(f"→ HEIF files are {compression_ratio:.2f}x SMALLER than COG")
    else:
        print(f"→ COG files are {1/compression_ratio:.2f}x SMALLER than HEIF")
else:
    print("\n⚠️  No results available for analysis.")

## 21. Export Final Report

In [ ]:
# Generate a comprehensive report
if final_results is not None and not final_results.empty:
    report_path = os.path.join(CONFIG['output_dir'], 'benchmark_report.txt')
    
    with open(report_path, 'w') as f:
        f.write("="*70 + "\n")
        f.write("COG vs GeoHEIF PERFORMANCE BENCHMARK REPORT\n")
        f.write("="*70 + "\n\n")
        
        f.write("CONFIGURATION:\n")
        f.write("-" * 70 + "\n")
        f.write(json.dumps(CONFIG, indent=2) + "\n\n")
        
        f.write("SUMMARY STATISTICS:\n")
        f.write("-" * 70 + "\n")
        f.write(str(final_results.groupby(['format', 'operation']).agg({
            'elapsed_time': ['mean', 'std'],
            'cpu_percent': 'mean',
            'memory_mb': 'mean',
            'bytes_read': 'mean',
            'file_size_mb': 'mean'
        }).round(4)) + "\n\n")
        
        f.write("DETAILED RESULTS:\n")
        f.write("-" * 70 + "\n")
        f.write(final_results.to_string() + "\n")
    
    print(f"\n✓ Saved comprehensive report to {report_path}")
    print(f"\n{'='*60}")
    print("BENCHMARK COMPLETE")
    print(f"{'='*60}")
    print(f"\nResults saved to: {CONFIG['output_dir']}")
    print(f"  - benchmark_results.csv")
    print(f"  - benchmark_report.txt")
    print(f"  - benchmark_*.png (plots)")
    print(f"\nNote: HEIF pyramids are embedded in the HEIF container.")
    print(f"      Access them using pillow_heif by image index.")
    print(f"      Pyramid metadata is stored in .pyramids.json sidecar files.")
else:
    print("\n⚠️  No results to export.")